# 🚀 DENSO VisionMind — Dual AI Serving: Qwen2.5-VL & ColPali Engine (Kaggle 2 x GPU T4)

Notebook này triển khai **HỆ THỐNG AI ĐA PHƯƠNG THỨC KÉP** trên **2 GPU T4** của Kaggle:
- 🧠 **GPU 0 (`cuda:0`)**: **vLLM Engine** phục vụ **Qwen2.5-VL-7B-Instruct-AWQ** (OpenAI-compatible API trên cổng nội bộ 8002).
- 👁️ **GPU 1 (`cuda:1`)**: **ColPali Engine** (`vidore/colpali-v1.2` trên nền PaliGemma-3B bfloat16) phục vụ **Late Interaction Multi-Vector MaxSim Search** từ pixel bản vẽ CAD & sơ đồ PDF.
- 🌐 **FastAPI Gateway (Cổng 8000)**: Tổng hợp cả 2 dịch vụ (`/v1/...` và `/colpali/...`) và expose qua 1 đường hầm **ngrok** duy nhất.

### ⚠️ YÊU CẦU BẮT BUỘC TRƯỚC KHI CHẠY:
Ở bảng **Settings** góc phải màn hình Kaggle:
1. **Accelerator**: Chọn **GPU T4 x 2**.
2. **Internet**: Bật **ON**.

In [ ]:
# Cell 1: Kiểm tra phần cứng 2 GPU T4 trên Kaggle
import torch

print("🔍 Đang kiểm tra GPU...")
if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
    print(f"⚠️ Cảnh báo: Số GPU hiện có là {torch.cuda.device_count() if torch.cuda.is_available() else 0}.")
    print("👉 Hãy nhìn sang góc phải màn hình Kaggle: Settings -> Accelerator -> Chọn 'GPU T4 x 2' để có đủ 2 GPU!")
else:
    print(f"✅ Đã sẵn sàng 2 GPU: GPU 0 = {torch.cuda.get_device_name(0)} | GPU 1 = {torch.cuda.get_device_name(1)}")
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

In [ ]:
# Cell 2: Cài đặt các thư viện vLLM, ColPali, FastAPI, uvicorn và pyngrok
# Gỡ bỏ torchao cũ (0.10.0) trên Kaggle để tránh xung đột với peft khi nạp ColPali
!pip uninstall -y torchao
!pip install -q -U vllm pyngrok colpali-engine fastapi uvicorn httpx python-multipart
print("✅ Cài đặt môi trường thành công!")

# CPU contract layout/OCR and BGE routes; install before launching gateway.
!pip install -q "surya-ocr==0.22.0" "numpy<2"
!pip install -q "docling-core>=2.48,<3" "transformers>=4.56,<5"
!pip check


In [ ]:
# Cell 3: Cấu hình Hugging Face Token & ngrok Authtoken
import os
from pyngrok import ngrok

HF_TOKEN = os.getenv("HF_TOKEN", "")
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

NGROK_AUTH_TOKEN = os.getenv("NGROK_AUTH_TOKEN", "")
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

print("✅ Đã cấu hình xác thực thành công cả HuggingFace và ngrok!")

A3 OCR, BGE and Granite run locally in Docker. This Kaggle gateway hosts Qwen and ColPali only.


In [ ]:

# Cell 5: Khởi động song song 2 AI Engine & Mở cổng ngrok Tunnel
import os
import subprocess
import time
import gc
import torch
import requests
from pyngrok import ngrok

# 1. Dọn dẹp tiến trình cũ & giải phóng triệt để VRAM trên cả 2 GPU
print("🧹 Đang giải phóng cổng mạng và VRAM trên cả 2 GPU...")
current_pid = os.getpid()
for p in subprocess.getoutput("nvidia-smi --query-compute-apps=pid --format=csv,noheader").split():
    if p.strip().isdigit() and int(p.strip()) != current_pid:
        try:
            os.kill(int(p.strip()), 9)
        except Exception:
            pass

!fuser -k 8000/tcp || true
!fuser -k 8002/tcp || true
!pkill -9 -f "vllm" || true
!pkill -9 -f "uvicorn" || true
time.sleep(2)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# 2. Khởi động vLLM Server chạy trên GPU 0 (cổng nội bộ 8002)
print("🚀 [GPU 0] Khởi động vLLM Qwen2.5-VL-7B-Instruct-AWQ trên cổng 8002...")
vllm_cmd = (
    "CUDA_VISIBLE_DEVICES=0 vllm serve Qwen/Qwen2.5-VL-7B-Instruct-AWQ "
    "--port 8002 "
    f"--gpu-memory-utilization {os.getenv('QWEN_GPU_MEMORY_UTILIZATION', '0.90')} "
    f"--max-model-len {int(os.getenv('QWEN_MAX_MODEL_LEN', '32768'))} "
    "--max-num-seqs 1 --enable-chunked-prefill --max-num-batched-tokens 2048 "
    "--enforce-eager "
    "--trust-remote-code "
    "--dtype half"
)
vllm_proc = subprocess.Popen(vllm_cmd, shell=True)

# 3. Khởi động Gateway FastAPI chạy ColPali trên GPU 1 (cổng 8000)
print("👁️ [GPU 1] Khởi động Gateway ColPali v1.2 trên cổng 8000...")
gateway_cmd = "uvicorn gateway_server:app --host 0.0.0.0 --port 8000"
gateway_proc = subprocess.Popen(gateway_cmd, shell=True)

# 4. Đợi cả 2 mô hình khởi động xong vào 2 GPU
print("⏳ Đang đợi cả 2 mô hình nạp vào VRAM (thường mất khoảng 60-90 giây)...")
vllm_ready = False
colpali_ready = False

for attempt in range(40):
    time.sleep(5)
    if not colpali_ready:
        try:
            r = requests.get("http://127.0.0.1:8000/colpali/health", timeout=2)
            if r.status_code == 200:
                colpali_ready = True
                print("  ✅ [GPU 1] ColPali Engine đã sẵn sàng!")
        except Exception:
            pass

    if not vllm_ready:
        try:
            r = requests.get("http://127.0.0.1:8002/v1/models", timeout=2)
            if r.status_code == 200:
                vllm_ready = True
                print("  ✅ [GPU 0] vLLM Qwen2.5-VL đã sẵn sàng!")
        except Exception:
            pass

    if vllm_ready and colpali_ready:
        break

if not (vllm_ready and colpali_ready):
    print("⚠️ Lưu ý: Một trong hai engine chưa phản hồi sau thời gian chờ. Đang tiếp tục mở tunnel...")

# 5. Mở ngrok Tunnel cho Gateway duy nhất (cổng 8000)
print("🌐 Đang mở đường hầm ngrok công khai qua Gateway (cổng 8000)...")
ngrok.kill()
tunnel = ngrok.connect(8000)
public_url = tunnel.public_url

print("\n" + "="*70)
print("🎉 ĐÃ TRIỂN KHAI THÀNH CÔNG HỆ THỐNG DUAL VISIONMIND!")
print(f"🔗 Public Gateway URL: {public_url}")
print(f"👉 VLLM_BASE_URL:     {public_url}/v1")
print(f"👉 COLPALI_BASE_URL:  {public_url}/colpali")
print("="*70)


In [ ]:
# Cell 6: Kiểm tra thử nghiệm truy vấn trực tiếp trên Kaggle
import requests

print("🧪 Kiểm tra ColPali:")
try:
    r_col = requests.get("http://127.0.0.1:8000/colpali/health", timeout=3)
    print("ColPali status:", r_col.json())
except Exception as e:
    print("ColPali check err:", e)

print("\n🧪 Kiểm tra vLLM Models:")
try:
    r_vllm = requests.get("http://127.0.0.1:8000/v1/models", timeout=3)
    print("vLLM models:", r_vllm.json())
except Exception as e:
    print("vLLM check err:", e)

Contract A3 routes require surya-ocr==0.22.0, paddleocr>=2.7,<3, paddlepaddle>=2.6,<3, numpy<2. Install those in the gateway environment before starting the app. Set RAG_INFERENCE_URL in the web root .env to the gateway base URL. BGE models download lazily on CPU. ColPali and Qwen retain their existing routes.

## Raise the active Qwen context without restarting ColPali
Run the following cell in the same kernel that started `vllm_proc`. It reuses the cached checkpoint, keeps the existing gateway/tunnel, and verifies `max_model_len` before reporting success.


In [ ]:
# Paste this entire file into a NEW cell in your running Kaggle notebook.
# Reuses the downloaded Qwen checkpoint; leaves ColPali and ngrok running.
import os
import subprocess
import time
from pathlib import Path

import psutil
import requests

context_tokens = int(os.getenv('QWEN_MAX_MODEL_LEN', '32768'))
model_id = 'Qwen/Qwen2.5-VL-7B-Instruct-AWQ'
previous = globals().get('vllm_proc')
if previous is None:
    raise RuntimeError('Run the updated model-start cell first: no vllm_proc handle in this notebook kernel.')

# Stop only the Qwen process tree created by this notebook.
if previous.poll() is None:
    parent = psutil.Process(previous.pid)
    children = parent.children(recursive=True)
    for process in children + [parent]:
        try:
            process.terminate()
        except psutil.NoSuchProcess:
            pass
    _, alive = psutil.wait_procs(children + [parent], timeout=10)
    for process in alive:
        try:
            process.kill()
        except psutil.NoSuchProcess:
            pass
    psutil.wait_procs(alive, timeout=10)

command = [
    'vllm', 'serve', model_id, '--port', '8002',
    '--gpu-memory-utilization', os.getenv('QWEN_GPU_MEMORY_UTILIZATION', '0.90'),
    '--max-model-len', str(context_tokens), '--max-num-seqs', '1',
    '--enable-chunked-prefill', '--max-num-batched-tokens', '2048',
    '--enforce-eager', '--trust-remote-code', '--dtype', 'half',
]
environment = dict(os.environ, CUDA_VISIBLE_DEVICES='0', HF_HUB_OFFLINE='1')
log_path = Path('/kaggle/working/qwen-long-context.log')
with log_path.open('w') as qwen_log:
    vllm_proc = subprocess.Popen(command, env=environment, stdout=qwen_log,
                                 stderr=subprocess.STDOUT, start_new_session=True)
print(f'Starting Qwen: context={context_tokens}, GPU=0. Log: {log_path}', flush=True)
ready = False
for attempt in range(90):
    if vllm_proc.poll() is not None:
        raise RuntimeError('Qwen startup failed. No context downgrade applied.\n' + log_path.read_text()[-6000:])
    try:
        models = requests.get('http://127.0.0.1:8002/v1/models', timeout=3)
        models.raise_for_status()
        data = models.json()['data']
        active = next((m for m in data if m.get('id') == model_id), {})
        if active.get('max_model_len', 0) >= context_tokens:
            print(f"Qwen ready: max_model_len={active['max_model_len']}", flush=True)
            ready = True
            break
    except (requests.RequestException, KeyError, ValueError):
        pass
    if attempt % 6 == 0:
        print(f'Waiting for Qwen... {attempt * 5}s', flush=True)
    time.sleep(5)
if not ready:
    raise RuntimeError('Qwen not ready; inspect qwen-long-context.log. No success claimed.')
print('ColPali and the existing public gateway were not restarted.', flush=True)
